# Chinook SQL Agent 完整工作流

本 notebook 按作业要求实现完整的 SQL Agent：

1. 数据库解析
2. Table 选择
3. SQL 生成
4. SQL 校验
5. SQL 改写
6. SQL 执行
7. 结果汇总

```text
中文问题
  -> 选表（LLM + 外键图补全）
  -> SQL 生成（schema + 样例 + 值域 + few-shot）
  -> 校验（sqlglot 静态 + EXPLAIN 动态 + 只读白名单）
  -> 失败则改写（错误反思循环，最多 N 次）
  -> 只读执行（超时 + 限行）
  -> 中文结果汇总
```

运行前请在项目根目录 `.env` 中配置 `DEEPSEEK_API_KEY`。

In [1]:
import sys
from pathlib import Path

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "sql_agent").exists():
    PROJECT_DIR = PROJECT_DIR.parent
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import pandas as pd
from IPython.display import Markdown, display

from sql_agent.catalog import DBCatalog
from sql_agent.config import settings
from sql_agent.llm import LLM

print("数据库:", settings.db_path)
print("模型:", settings.model)
print("API Key 已配置:", bool(settings.api_key))
assert settings.api_key, "请在 .env 中配置 DEEPSEEK_API_KEY"

llm = LLM(settings.api_key, settings.base_url, settings.model)

数据库: D:\BaiduNetdiskDownload\ai\直播课程\十一周\第11周-ChatBI数据智能问答\Week12\04_SQL-Code-Agent-Demo\chinook.db
模型: deepseek-chat
API Key 已配置: True


## ① 数据库解析

采集表、字段、类型、主键、外键、行数、样例数据、字段取值分布，并生成提示词用的 DDL。

In [2]:
catalog = DBCatalog(settings.db_path, settings.cache_dir).load(force=True)

print(f"共解析到 {len(catalog.table_names)} 张业务表")
for name in catalog.table_names:
    print(f"- {name}: {catalog.tables[name].row_count} 行")

fks = [
    {
        "主表": fk.table,
        "外键列": ",".join(fk.columns),
        "引用表": fk.referred_table,
        "引用列": ",".join(fk.referred_columns),
    }
    for fk in catalog.foreign_keys
]
display(pd.DataFrame(fks))
display(Markdown("```sql\n" + catalog.schema_text(["tracks"]) + "\n```"))

共解析到 11 张业务表


- albums: 347 行
- artists: 275 行
- customers: 59 行
- employees: 8 行
- genres: 25 行
- invoice_items: 2240 行
- invoices: 412 行
- media_types: 5 行
- playlist_track: 8715 行
- playlists: 18 行
- tracks: 3503 行


,主表,外键列,引用表,引用列
0,albums,ArtistId,artists,ArtistId
1,customers,SupportRepId,employees,EmployeeId
2,employees,ReportsTo,employees,EmployeeId
3,invoice_items,TrackId,tracks,TrackId
4,invoice_items,InvoiceId,invoices,InvoiceId
5,invoices,CustomerId,customers,CustomerId
6,playlist_track,TrackId,tracks,TrackId
7,playlist_track,PlaylistId,playlists,PlaylistId
8,tracks,MediaTypeId,media_types,MediaTypeId
9,tracks,GenreId,genres,GenreId


```sql
-- tracks (3503 行)
CREATE TABLE "tracks" (
  "TrackId" INTEGER PRIMARY KEY,,
  "Name" NVARCHAR(200),,
  "AlbumId" INTEGER,  -- FK -> albums.AlbumId,
  "MediaTypeId" INTEGER,  -- FK -> media_types.MediaTypeId,
  "GenreId" INTEGER,  -- FK -> genres.GenreId,
  "Composer" NVARCHAR(220),,
  "Milliseconds" INTEGER,,
  "Bytes" INTEGER,,
  "UnitPrice" NUMERIC(10,2)
);

样例行:
| TrackId | Name | AlbumId | MediaTypeId | GenreId | Composer | Milliseconds | Bytes | UnitPrice |
|---|---|---|---|---|---|---|---|---|
| 1 | For Those About To Rock (We Salute You) | 1 | 1 | 1 | Angus Young, Malcolm Young, Brian Johnson | 343719 | 11170334 | 0.99 |
| 2 | Balls to the Wall | 2 | 2 | 1 | NULL | 342562 | 5510424 | 0.99 |
| 3 | Fast As a Shark | 3 | 2 | 1 | F. Baltes, S. Kaufman, U. Dirkscneider & W. Hoffman | 230619 | 3990994 | 0.99 |

值域提示:
- TrackId: 范围 [1, 3503]
- AlbumId: 范围 [1, 347]
- MediaTypeId: 范围 [1, 5]
- GenreId: 范围 [1, 25]
- Milliseconds: 范围 [1071, 5286953]
- Bytes: 范围 [38747, 1059546140]
- UnitPrice: 范围 [0.99, 1.99]
```

## ② Table 选择

关键词提示 + LLM 选表，再用外键图补全多对多中间表。

In [3]:
from sql_agent.selector import TableSelector

question = "哪个歌单包含最多的摇滚歌曲？"
selector = TableSelector(catalog, llm)
selection = selector.select(question)

print("问题:", question)
print("选中表:", selection.tables)
print("理由:", selection.reason)
print("是否兜底:", selection.used_fallback)

问题: 哪个歌单包含最多的摇滚歌曲？
选中表: ['genres', 'playlist_track', 'playlists', 'tracks']
理由: 需要连接歌单、歌单曲目中间表、曲目和流派表，按摇滚流派统计各歌单的歌曲数量。
是否兜底: False


## ③ SQL 生成

使用选中表的 DDL、样例行、值域提示和 few-shot 示例生成 SQLite SQL。

In [4]:
from sql_agent.generator import SQLGenerator

schema = catalog.schema_text(selection.tables)
generator = SQLGenerator(llm)
sql = generator.generate(question, schema)
print(sql)

SELECT p.Name AS playlist, COUNT(*) AS rock_tracks
FROM playlists p
JOIN playlist_track pt ON p.PlaylistId = pt.PlaylistId
JOIN tracks t ON pt.TrackId = t.TrackId
JOIN genres g ON t.GenreId = g.GenreId
WHERE g.Name = 'Rock'
GROUP BY p.Name
ORDER BY rock_tracks DESC
LIMIT 1;


## ④ SQL 校验

静态：sqlglot 解析，检查单语句、只读类型、表/列是否存在；动态：EXPLAIN 编译校验。

In [5]:
from sql_agent.validator import SQLValidator

validator = SQLValidator(catalog)

result = validator.validate(sql)
print("通过:", result.ok)
print("阶段:", result.stage)
print("信息:", result.error)

# 反例：不存在的列
bad_validation = validator.validate("SELECT NotAColumn FROM tracks")
print("\n反例通过:", bad_validation.ok)
print("反例阶段:", bad_validation.stage)
print("反例信息:", bad_validation.error)

通过: True
阶段: ok
信息: 静态/动态校验通过

反例通过: False
反例阶段: unknown_column
反例信息: 列不存在: NotAColumn


## ⑤ SQL 改写

把「问题 + 原 SQL + 错误信息 + 历史尝试」交给大模型修复，修复后重新校验。

In [6]:
from sql_agent.rewriter import SQLRewriter

rewriter = SQLRewriter(llm)

rewrite_question = "一共有多少首歌曲？"
rewrite_schema = catalog.schema_text(["tracks"])
bad_sql = "SELECT Foo FROM tracks"

bad_val = validator.validate(bad_sql)
print("原始 SQL:", bad_sql)
print("错误:", bad_val.stage, bad_val.error)

fixed_sql = rewriter.rewrite(
    rewrite_question,
    rewrite_schema,
    bad_sql,
    f"{bad_val.stage}: {bad_val.error}",
    history=[],
)
print("\n改写后 SQL:")
print(fixed_sql)
print("\n再次校验:", validator.validate(fixed_sql))

原始 SQL: SELECT Foo FROM tracks
错误: unknown_column 列不存在: Foo



改写后 SQL:
SELECT COUNT(*) FROM tracks

再次校验: ValidationResult(ok=True, stage='ok', error='静态/动态校验通过')


## ⑥ SQL 执行

只读连接 + 超时中断 + 最多返回 N 行，避免写操作和超大结果。

In [7]:
from sql_agent.executor import SQLExecutor

executor = SQLExecutor(settings.db_path, max_rows=50, timeout_s=settings.query_timeout_s)
exec_result = executor.execute(sql)

print("执行成功:", exec_result.ok)
print("返回行数:", exec_result.row_count)
print("是否截断:", exec_result.truncated)
print("耗时(ms):", round(exec_result.elapsed_ms, 2))

if exec_result.ok:
    display(pd.DataFrame(exec_result.rows, columns=exec_result.columns))
else:
    print("错误:", exec_result.error)

执行成功: True
返回行数: 1
是否截断: False
耗时(ms): 1.59


,playlist,rock_tracks
0,Music,2594


## ⑦ 结果汇总

把问题、SQL 和结果表交给大模型，生成中文结论；禁止编造数字。

In [8]:
from sql_agent.summarizer import ResultSummarizer

summarizer = ResultSummarizer(llm)
answer = summarizer.summarize(question, sql, exec_result)
print(answer)

包含摇滚歌曲最多的歌单是 **Music**，共有 **2594 首**摇滚歌曲。


## 完整 Agent 端到端

一次调用串起七步，并保留每一步的执行轨迹（Trace）。

In [9]:
from sql_agent.agent import SQLAgent

agent = SQLAgent(catalog=catalog, llm=llm)
agent_result = agent.run("消费总额最高的前5位客户是谁？")

for step in agent_result.steps:
    print(f"[{step.status}] {step.name} ({step.elapsed_ms:.0f} ms)")
    print(step.detail)
    print("-" * 60)

print("最终 SQL:")
print(agent_result.sql)
print("\n答案:")
print(agent_result.answer)
display(agent_result.to_dataframe())

[ok] ② 表选择 (762 ms)
选中表: customers, invoices
理由: 需要从客户表获取客户信息，并从发票表按客户汇总消费总额排序取前5。
------------------------------------------------------------
[ok] ③ SQL 生成 (679 ms)
SELECT c.FirstName || ' ' || c.LastName AS customer, ROUND(SUM(i.Total), 2) AS total_spent
FROM invoices i
JOIN customers c ON i.CustomerId = c.CustomerId
GROUP BY c.CustomerId
ORDER BY total_spent DESC
LIMIT 5;
------------------------------------------------------------
[ok] ④ SQL 校验 (2 ms)
静态/动态校验通过
------------------------------------------------------------
[ok] ⑥ SQL 执行 (0 ms)
返回 5 行，耗时 0.4 ms
------------------------------------------------------------
[ok] ⑦ 结果汇总 (1082 ms)
消费总额最高的前5位客户依次为：

1. Helena Holý：49.62
2. Richard Cunningham：47.62
3. Luis Rojas：46.62
4. Ladislav Kovács：45.62
5. Hugh O'Reilly：45.62

以上金额为各客户消费总额（按发票金额汇总，保留两位小数）。
------------------------------------------------------------
最终 SQL:
SELECT c.FirstName || ' ' || c.LastName AS customer, ROUND(SUM(i.Total), 2) AS total_spent
FROM invoices i
JOIN 

,customer,total_spent
0,Helena Holý,49.62
1,Richard Cunningham,47.62
2,Luis Rojas,46.62
3,Ladislav Kovács,45.62
4,Hugh O'Reilly,45.62


## 批量评测

`eval/gold.jsonl` 包含 30 条中文问题、gold SQL 和期望表，指标包括 SQL 成功率、执行准确率、选表召回率。

In [10]:
# 完整评测：python eval/evaluate.py
# 下面只跑前 5 条，便于快速验证
import subprocess

completed = subprocess.run(
    [sys.executable, "eval/evaluate.py", "--limit", "5"],
    capture_output=True,
    text=True,
    encoding="utf-8",
)
print(completed.stdout[-3000:])
if completed.stderr:
    print(completed.stderr[-1000:])

[01/05] PASS q01 recall=1.00 attempts=1 3.5s | 数据库里一共有多少首歌曲？
[02/05] PASS q02 recall=1.00 attempts=1 2.4s | 一共有多少位艺人？
[03/05] PASS q03 recall=1.00 attempts=1 2.5s | 一共有多少张专辑？
[04/05] PASS q04 recall=1.00 attempts=1 2.7s | 每个流派有多少首歌曲？按歌曲数量从高到低排序。
[05/05] PASS q05 recall=1.00 attempts=1 2.2s | 歌曲数量最多的前5个流派是哪些？

案例数        : 5
SQL 成功      : 5/5 (100.0%)
校验通过      : 5/5 (100.0%)
执行准确率    : 5/5 (100.0%)
平均选表召回  : 100.0%
平均尝试次数  : 1.0
平均耗时      : 2.672 s

详细结果已保存: D:\BaiduNetdiskDownload\ai\直播课程\十一周\第11周-ChatBI数据智能问答\Week12\06_SQL-Agent-Project\eval\results.json



## 启动可视化 UI

```bash
streamlit run app.py
```

UI 支持：聊天式提问、步骤轨迹、SQL 查看、结果表格、自动柱状图、示例问题。